In [7]:
import pandas as pd
import numpy as np


# Topography per Italian region
- Clip SRTM 30 m DEM to Italy with limits_IT_regions.geojson
- Compute elevation spread (mean/median/std/range/IQR) and slope-based ruggedness per region
- Export metrics to CSV plus an enriched GeoJSON

In [ ]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np
from rasterstats import zonal_stats
import rasterio
from rasterio.merge import merge
import requests
import gzip
import shutil

data_dir = Path("c:/Users/juanx/Documents/GitHub/juanxgi83.github.io/data")
regions_path = data_dir / "limits_IT_regions.geojson"
work_dir = Path("C:/Users/juanx/Documents/PP434/topography_cache")
tile_dir = work_dir / "srtm_tiles"
work_dir.mkdir(parents=True, exist_ok=True)
tile_dir.mkdir(parents=True, exist_ok=True)

gdf = gpd.read_file(regions_path).to_crs("EPSG:4326")
print("Region columns:", [c for c in gdf.columns if c != gdf.geometry.name])

def tile_names(bounds):
    minx, miny, maxx, maxy = bounds
    lons = range(int(np.floor(minx)), int(np.ceil(maxx)))
    lats = range(int(np.floor(miny)), int(np.ceil(maxy)))
    names = []
    for lat in lats:
        for lon in lons:
            ns = "N" if lat >= 0 else "S"
            ew = "E" if lon >= 0 else "W"
            names.append(f"{ns}{abs(lat):02d}{ew}{abs(lon):03d}")
    return names

def download_and_extract(tile):
    folder = tile[:3]
    url = f"https://elevation-tiles-prod.s3.amazonaws.com/skadi/{folder}/{tile}.hgt.gz"
    gz_path = tile_dir / f"{tile}.hgt.gz"
    hgt_path = tile_dir / f"{tile}.hgt"
    if hgt_path.exists():
        return hgt_path
    if not gz_path.exists():
        print("Downloading", tile)
        resp = requests.get(url, timeout=60)
        resp.raise_for_status()
        gz_path.write_bytes(resp.content)
    with gzip.open(gz_path, "rb") as f_in, open(hgt_path, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)
    return hgt_path

bounds = gdf.total_bounds
tiles = tile_names(bounds)
print(f"Need {len(tiles)} tiles; downloading/mosaicking...")
tile_paths = [download_and_extract(t) for t in tiles]

srcs = [rasterio.open(p) for p in tile_paths]
mosaic, out_transform = merge(srcs)
for src in srcs:
    src.close()

profile = srcs[0].profile
profile.update({"height": mosaic.shape[1], "width": mosaic.shape[2], "transform": out_transform, "driver": "GTiff", "nodata": -32768})
dem_path = work_dir / "italy_srtm1.tif"
with rasterio.open(dem_path, "w", **profile) as dst:
    dst.write(mosaic)

print("DEM saved to", dem_path)
dem_path

Region columns: ['reg_name', 'reg_istat_code_num', 'reg_istat_code']
Need 169 tiles; downloading/mosaicking...
DEM saved to c:\Users\juanx\Documents\GitHub\juanxgi83.github.io\data\topography_cache\italy_srtm1.tif


WindowsPath('c:/Users/juanx/Documents/GitHub/juanxgi83.github.io/data/topography_cache/italy_srtm1.tif')

In [ ]:
nodata_val = -9999.0
slope_path = work_dir / "italy_slope_deg.tif"

with rasterio.open(dem_path) as src:
    dem = src.read(1, masked=True).astype("float32")
    dem = dem.filled(np.nan)
    transform = src.transform
    xres = transform.a
    yres = -transform.e
    gy, gx = np.gradient(dem, yres, xres)
    slope_rad = np.arctan(np.sqrt(gx**2 + gy**2))
    slope_deg = np.degrees(slope_rad)
    profile = src.profile
    profile.update(dtype="float32", nodata=nodata_val)
    slope_clean = np.where(np.isfinite(slope_deg), slope_deg, nodata_val).astype("float32")
    with rasterio.open(slope_path, "w", **profile) as dst:
        dst.write(slope_clean, 1)

def _finite(arr):
    arr = np.asarray(arr)
    arr = arr[np.isfinite(arr)]
    return arr

def p25(arr):
    arr = _finite(arr)
    return np.percentile(arr, 25) if arr.size else np.nan

def p75(arr):
    arr = _finite(arr)
    return np.percentile(arr, 75) if arr.size else np.nan

elev_stats = zonal_stats(
    gdf,
    dem_path,
    stats=["mean", "median", "std", "min", "max"],
    nodata=nodata_val,
    add_stats={"p25": p25, "p75": p75}
)

slope_stats = zonal_stats(
    gdf,
    slope_path,
    stats=["mean", "std", "max"],
    nodata=nodata_val
)

elev_df = pd.DataFrame(elev_stats).add_prefix("elev_")
slope_df = pd.DataFrame(slope_stats).add_prefix("slope_")
result = pd.concat([gdf.reset_index(drop=True), elev_df, slope_df], axis=1)
result["elev_range"] = result["elev_max"] - result["elev_min"]
result["elev_iqr"] = result["elev_p75"] - result["elev_p25"]

region_col_candidates = [c for c in gdf.columns if c != gdf.geometry.name and c.lower().startswith(("name", "reg"))]
region_col = region_col_candidates[0] if region_col_candidates else [c for c in gdf.columns if c != gdf.geometry.name][0]

out_csv = work_dir / "italy_region_topography_metrics.csv"
result[[region_col, "elev_mean", "elev_median", "elev_std", "elev_range", "elev_iqr", "elev_p25", "elev_p75", "slope_mean", "slope_std", "slope_max"]].to_csv(out_csv, index=False)

out_geojson = work_dir / "limits_IT_regions_with_topography.geojson"
result.to_file(out_geojson, driver="GeoJSON")

print("Saved:", out_csv)
print("Saved:", out_geojson)
result[[region_col, "elev_mean", "elev_range", "slope_mean"]].head()

TypeError: Cannot convert fill_value nan to dtype int16